# 05 · Final model: multi-temporal DeepLabV3+

**Why this model.** Notebooks 02 and 03 trained nine single-month models (0.08–75 M parameters, three architectures,
four backbones). On unseen regions they all reach 0.25–0.30 test mIoU, and none beats DeepLabV3+ · ResNet50, while
the same type of model reaches 0.56–0.58 on the training regions in 2025–26. The conclusion was that **architecture is
not the bottleneck**; the main confusions (crops ↔ rangeland, bare ↔ snow/ice, trees ↔ built-up between seasons)
are between classes that differ mainly in **how they change over the year**, which a single monthly image cannot show.

**What changes: only the input.** For a target month the model sees that month **and the two previous months**
(stacked as 3 × 3 = 9 channels: VV, VH, VV−VH per month). Example: the July map is made from March + May + July.
Everything else is identical to the notebook-02 model, so a difference can be attributed to the temporal information:

| | Notebook 02 (reference) | This notebook |
|---|---|---|
| Architecture | DeepLabV3+ · ResNet50 (ImageNet) | same; the pretrained first-layer filters are copied to each month |
| Input | 1 month (3 channels) | 3 months (9 channels) |
| Data | all train regions, 2023–24 | same patches and labels (label = target month) |
| Training | AdamW 3e-4, batch 16, weighted CE + Dice, augmentation, early stopping | same |

**Fair comparison.** Both models are evaluated on exactly the same patches: those whose two previous months exist.
For the test regions the export starts in January 2025, so January and March 2025 have no earlier months and are
left out for both models. Nothing new is downloaded: the earlier months come from the existing tiles.

In [ ]:
import os, shutil, subprocess
if shutil.which("nvidia-smi"):
    free = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout.split()
    os.environ["CUDA_VISIBLE_DEVICES"] = str(max(range(len(free)), key=lambda i: int(free[i])))
    print("Free GPU memory (MB):", free, "-> using GPU", os.environ["CUDA_VISIBLE_DEVICES"])
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
keras.mixed_precision.set_global_policy("mixed_float16")
keras.utils.set_random_seed(42)

from src import config as C, model as M, evaluation as E, patching as P, temporal as TM, training as T

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
SMOKE_TEST = False     # True: 64 random patches per split, 2 epochs - only checks that the code runs

N_TIMES = TM.N_TIMES   # 3 months per input
BATCH = 16             # as in notebook 02
LR = 3e-4
MAX_EPOCHS = 2 if SMOKE_TEST else 50
PATIENCE = 8

WEIGHTS_DIR = Path(os.environ.get("LULC_WEIGHTS_DIR", ROOT / "weights"))
OUT_DIR = WEIGHTS_DIR / ("multitemporal_smoke_test" if SMOKE_TEST else "multitemporal")
OUT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE = ROOT / "weights" / "deeplabv3plus.keras"      # single-month DeepLabV3+ ResNet50 from notebook 02
stats = M.load_stats()

## 1. Multi-temporal patches

For every patch of notebooks 01/02, the same 256 × 256 window is read from the two previous exported months of the
same study area (2–3 months apart). Patches whose earlier months are missing or contain no-data pixels are dropped.

In [ ]:
sizes = {}
for split in ["train", "val", "test"]:
    if not (C.PATCH_DIR / f"meta_{split}_mt{N_TIMES}.csv").exists():
        TM.build_split(split, n_times=N_TIMES)
    single = len(pd.read_csv(C.PATCH_DIR / f"meta_{split}.csv"))
    multi = pd.read_csv(C.PATCH_DIR / f"meta_{split}_mt{N_TIMES}.csv")
    sizes[split] = {"single-month patches": single, "multi-temporal patches": len(multi),
                    "kept %": round(100 * len(multi) / single, 1), "regions": multi.aoi.nunique(),
                    "target months": multi.month.nunique()}
pd.DataFrame(sizes).T

In [ ]:
def load_mt(split, cache=not SMOKE_TEST):
    # memory-mapped; on the server the training array is first copied to the fast local disk
    path = C.PATCH_DIR / f"X_{split}_mt{N_TIMES}.npy"
    if cache and Path("/tmp").exists():
        local = Path("/tmp/lulc_cache") / path.name
        if not local.exists():
            local.parent.mkdir(exist_ok=True)
            shutil.copy(path, local)
        path = local
    X = np.load(path, mmap_mode="r")
    y = np.load(C.PATCH_DIR / f"y_{split}_mt{N_TIMES}.npy", mmap_mode="r")
    meta = pd.read_csv(C.PATCH_DIR / f"meta_{split}_mt{N_TIMES}.csv")
    if SMOKE_TEST:
        idx = np.sort(np.random.default_rng(0).choice(len(X), min(64, len(X)), replace=False))
        return np.asarray(X[idx]), np.asarray(y[idx]), meta.iloc[idx].reset_index(drop=True)
    return X, y, meta

X_train, y_train, meta_train = load_mt("train")
X_val, y_val, meta_val = load_mt("val", cache=False)
X_test, y_test, meta_test = load_mt("test", cache=False)
class_weights = M.median_frequency_weights(P.class_distribution(f"train_mt{N_TIMES}").pixels)
data = dict(X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val, stats=stats, class_weights=class_weights)
print(f"train {X_train.shape} | val {X_val.shape} | test {X_test.shape}")
print("Example input months:", meta_train.input_months.iloc[0], "->", meta_train.month.iloc[0])

## 2. Training

Identical procedure to notebook 02: augmentation (flips, rotations; brightness change drawn separately for each
month; speckle noise), weighted cross-entropy + Dice, AdamW, learning-rate halving after 3 and early stopping after 8
epochs without validation-mIoU improvement, best epoch restored.

In [ ]:
model_mt, history_mt, info_mt = T.train_model(f"deeplabv3plus_mt{N_TIMES}", data, OUT_DIR, lr=LR, batch=BATCH,
                                              max_epochs=MAX_EPOCHS, patience=PATIENCE)
T.plot_history(history_mt, f"DeepLabV3+ · ResNet50 · {N_TIMES} months")
info_mt

## 3. Comparison with the single-month model on the same patches

The single-month model gets only the target month (the last 3 channels), the multi-temporal model all 9.

In [ ]:
model_1 = keras.models.load_model(REFERENCE, compile=False)
last = slice(-len(C.S1_BANDS), None)
res = {
    "single month": {"val": T.evaluate(model_1, X_val[..., last], y_val, stats),
                     "test": T.evaluate(model_1, X_test[..., last], y_test, stats)},
    f"{N_TIMES} months": {"val": T.evaluate(model_mt, X_val, y_val, stats),
                          "test": T.evaluate(model_mt, X_test, y_test, stats)},
}
comparison = pd.DataFrame({name: {f"{split} {k}": r[split][k] for split in ["val", "test"] for k in ["OA", "mIoU", "mF1"]}
                           for name, r in res.items()}).T
comparison.loc["difference"] = comparison.iloc[1] - comparison.iloc[0]
comparison.round(4)

In [ ]:
per_class = pd.DataFrame({
    ("IoU", "single month"): res["single month"]["test"]["per_class"]["IoU"],
    ("IoU", f"{N_TIMES} months"): res[f"{N_TIMES} months"]["test"]["per_class"]["IoU"],
    ("F1", "single month"): res["single month"]["test"]["per_class"]["F1"],
    ("F1", f"{N_TIMES} months"): res[f"{N_TIMES} months"]["test"]["per_class"]["F1"],
})
per_class[("IoU", "difference")] = per_class[("IoU", f"{N_TIMES} months")] - per_class[("IoU", "single month")]
per_class.insert(0, ("test pixels", "%"), res["single month"]["test"]["per_class"]["support_%"])
per_class.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
for ax, (name, r) in zip(axes, res.items()):
    cm = r["test"]["cm"].astype(float)
    with np.errstate(invalid="ignore"):
        cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(C.CLASSES)), C.CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(len(C.CLASSES)), C.CLASSES)
    for i in range(len(C.CLASSES)):
        for j in range(len(C.CLASSES)):
            if np.isfinite(cmn[i, j]):
                ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted"); ax.set_ylabel("reference (Dynamic World)")
    ax.set_title(f"Test confusion matrix · {name}")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_single_vs_multitemporal.png", dpi=150)
plt.show()

In [ ]:
def by_region(r, meta):
    rows = {}
    for aoi in sorted(meta.aoi.unique()):
        m = (meta.aoi == aoi).values
        s = E.scores(E.confusion(y_test[m], r["test"]["pred"][m]))
        rows[aoi] = (s["OA"], s["mIoU"])
    return rows

r1, r3 = by_region(res["single month"], meta_test), by_region(res[f"{N_TIMES} months"], meta_test)
regions = pd.DataFrame({"OA single": {a: v[0] for a, v in r1.items()}, f"OA {N_TIMES} months": {a: v[0] for a, v in r3.items()},
                        "mIoU single": {a: v[1] for a, v in r1.items()}, f"mIoU {N_TIMES} months": {a: v[1] for a, v in r3.items()},
                        "patches": meta_test.aoi.value_counts()})
regions["Δ OA"] = regions[f"OA {N_TIMES} months"] - regions["OA single"]
regions.round(3)

## 4. Decision

Selection rule fixed before seeing any test result: the multi-temporal model replaces the single-month model in the
prototype (`weights/best_model.keras`) only if its **validation** mIoU on the same patches is higher. The prototype
detects the number of input months from the model and fetches the earlier months automatically.

In [ ]:
val_1, val_3 = res["single month"]["val"]["mIoU"], res[f"{N_TIMES} months"]["val"]["mIoU"]
use_mt = val_3 > val_1
print(f"Validation mIoU: single month {val_1:.4f} | {N_TIMES} months {val_3:.4f}")
if use_mt and not SMOKE_TEST:
    shutil.copy(OUT_DIR / f"deeplabv3plus_mt{N_TIMES}.keras", WEIGHTS_DIR / "best_model.keras")
    print(f"-> the {N_TIMES}-month model is now weights/best_model.keras (single-month model kept as weights/deeplabv3plus.keras)")
else:
    print("-> the prototype keeps the single-month model")

summary = {"n_times": N_TIMES, "selected": f"{N_TIMES} months" if use_mt else "single month", "training": info_mt,
           "comparison": comparison.round(4).to_dict(),
           "per_class_IoU": {k: v["test"]["per_class"]["IoU"].round(4).to_dict() for k, v in res.items()},
           "regions": regions.round(4).to_dict()}
(OUT_DIR / "multitemporal_results.json").write_text(json.dumps(summary, indent=2, default=float))

## Findings

*(Written after the run.)*